# Multiclass Fish Image Classification - Colab GPU version

This notebook runs **exactly the same code** as the project on your computer.
Nothing is copied or rewritten here: it downloads the project files, downloads
the dataset, and then calls `train_all_models.py` - the same script you would
run locally.

The only difference is the hardware. Colab gives you a free GPU, which trains
all six models in roughly 20 minutes instead of several hours on a laptop CPU.

**Before you start:** turn the GPU on.
`Runtime` menu -> `Change runtime type` -> `Hardware accelerator` -> **T4 GPU** -> `Save`.

Then run the cells below in order, top to bottom.

## Step 1 - check that the GPU is really on

If this prints `GPUs available: []`, the GPU is not switched on. Go back and
set the runtime type, then run this cell again. Do not carry on without it -
training on Colab's CPU is slower than your own laptop.

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

# nvidia-smi asks the graphics card to describe itself
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 2 - get the project code

Two ways. **Pick one** and run only that cell.

### Option A (recommended) - clone from your GitHub repository

The project has to be on a public GitHub repository anyway, so this is the
tidiest route. Paste your repository URL into `GITHUB_REPO_URL` below.

In [ ]:
GITHUB_REPO_URL = ""   # e.g. "https://github.com/your-username/multiclass_fish_classification"

if GITHUB_REPO_URL:
    !git clone $GITHUB_REPO_URL project
    %cd project
    !ls
else:
    print("No URL set. Either fill in GITHUB_REPO_URL above, or use Option B below.")

### Option B - upload the code as a zip

On your own computer, run this in the project folder:

```
python make_colab_bundle.py
```

That creates `colab_bundle.zip`. Run the cell below and choose that file when
the Upload button appears.

In [ ]:
import os

if not os.path.exists("project"):
    from google.colab import files

    print("Choose colab_bundle.zip ...")
    uploaded = files.upload()          # opens a file picker

    !mkdir -p project
    !unzip -q -o colab_bundle.zip -d project
    %cd project
    !ls
else:
    print("project/ already exists - you already used Option A.")

## Step 3 - install the one library Colab is missing

Colab already has TensorFlow, pandas, scikit-learn, matplotlib and seaborn.
It only needs `tabulate`, which lets pandas write the Markdown report table.

In [ ]:
!pip install -q tabulate

## Step 4 - download the dataset

261 MB from the project's Google Drive folder. This uses the
`drive.usercontent.google.com` address, which downloads at full speed. The
ordinary Drive download path is throttled and can take hours for the same file.

The zip already contains a `data/` folder split into `train`, `val` and `test`,
so unzipping it here puts everything exactly where `src/config.py` expects it.

In [ ]:
FILE_ID = "1fo4WUz1StqsYehpI2b167bGtAWnPLVl4"

!curl -L -o fish_dataset.zip "https://drive.usercontent.google.com/download?id=$FILE_ID&export=download&confirm=t"
!ls -lh fish_dataset.zip

In [ ]:
import zipfile, sys

# Safety check: refuse anything that is not a .jpg sitting inside data/
z = zipfile.ZipFile("fish_dataset.zip")
for name in z.namelist():
    if name.startswith("/") or ".." in name.split("/"):
        sys.exit("Unsafe path in zip: " + name)
    if not name.endswith("/") and not name.lower().endswith(".jpg"):
        sys.exit("Unexpected file in zip: " + name)

print("Zip looks fine. Extracting ...")
z.extractall(".")
print("Done.")

In [ ]:
# How many images did we get?
import os

for split in ["train", "val", "test"]:
    folder = os.path.join("data", split)
    total = 0
    print("\n" + split + ":")
    for species in sorted(os.listdir(folder)):
        count = len(os.listdir(os.path.join(folder, species)))
        total += count
        print("   {:<40} {:>6}".format(species, count))
    print("   {:<40} {:>6}".format("TOTAL", total))

## Step 5 - train all six models

This is the same `train_all_models.py` you would run on your own machine.
The `-u` flag just means "print progress as it happens" instead of buffering it.

On a T4 GPU expect roughly 15-25 minutes for all six models together.

For each model you will see the layer summary, a progress bar per epoch, the
EarlyStopping and checkpoint messages, then its accuracy, precision, recall
and F1-score.

In [ ]:
!python -u train_all_models.py

### Want a quick test first?

Run this instead, to train one model for two epochs and check everything works
end to end. Then come back and run the full cell above.

```
!python -u train_all_models.py --model MobileNet --epochs 2
```

## Step 6 - write the comparison report

Builds `outputs/reports/comparison_report.md` and fills in the Results section
of `README.md` automatically.

In [ ]:
!python make_report.py

## Step 7 - look at the results

The comparison chart first, then each model's training curves and confusion
matrix.

In [ ]:
from IPython.display import Image, display
import glob

display(Image("outputs/plots/model_comparison.png"))

for path in sorted(glob.glob("outputs/plots/*_training_history.png")):
    print(path)
    display(Image(path))

for path in sorted(glob.glob("outputs/plots/*_confusion_matrix.png")):
    print(path)
    display(Image(path))

In [ ]:
# The comparison table, and the per-species scores for the winner
import pandas as pd

table = pd.read_csv("outputs/reports/model_comparison.csv")
table = table.sort_values("accuracy", ascending=False)
print(table.to_string(index=False))

best_name = open("models/best_model_name.txt").read().strip()
print("\n" + "=" * 70)
print(open("outputs/reports/" + best_name + "_classification_report.txt").read())

## Step 8 - download the results to your computer

This zips up the trained models and every graph and report, then downloads it.

**Careful:** the six `.h5` files together are a few hundred MB, so the zip is
large and the download takes a minute or two.

In [ ]:
!zip -r -q results.zip models outputs
!ls -lh results.zip

from google.colab import files
files.download("results.zip")

### What to do with it

Unzip `results.zip` into your project folder on your computer, so you end up
with:

```
multiclass_fish_classification/
  models/best_model.h5          <- the winner
  models/class_names.json
  models/best_model_name.txt
  outputs/plots/...
  outputs/reports/...
```

Then run the web app locally:

```
streamlit run app.py
```

It loads `models/best_model.h5` and `models/class_names.json`, both of which
are in that zip, so it works straight away.

**Tip for your demo video:** record the Streamlit app, not this notebook.
Upload a fish photo, show the predicted species, the confidence percentage and
the bar chart across all eleven species. Then show the comparison chart and one
confusion matrix to explain how you chose the model.